# Text Classification: Naive Bayes from Scratch vs. Scikit-Learn

This notebook provides a hands-on exploration of **text classification using Multinomial Naive Bayes** — from a full NumPy from-scratch implementation to production-ready sklearn models.

We load the **IMDB movie review dataset** (50,000 reviews, binary sentiment), preprocess text, implement **Multinomial Naive Bayes using only NumPy**, compare with sklearn's `MultinomialNB` and `TfidfVectorizer + LogisticRegression`, and run hyperparameter experiments on smoothing alpha and vocabulary size.

## Prerequisites
- Bayes' theorem and conditional probability
- Basic probability theory (priors, likelihoods)
- Tokenization and bag-of-words representation
- sklearn basics (CountVectorizer, TfidfVectorizer)

## Dataset
[IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

**Credits:** Andrew L. Maas, Raymond E. Daly, Peter T. Pham, Dan Huang, Andrew Y. Ng, and Christopher Potts. (2011). *Learning Word Vectors for Sentiment Analysis.* ACL.

In [ ]:
import numpy as np              # Numerical operations and matrix math
import pandas as pd             # Data loading and manipulation
import matplotlib.pyplot as plt # Plotting
import seaborn as sns           # Statistical visualizations
import re                       # Text cleaning with regex
from collections import Counter # Vocabulary / frequency building
import warnings                 # Suppress deprecation warnings
warnings.filterwarnings('ignore')

# sklearn imports
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix)

np.random.seed(42)              # Reproducibility

## Part 1: Theory Recap — Naive Bayes for Text Classification

- **Bayes' Theorem:** P(class|document) ∝ P(class) × P(document|class). We compute the posterior probability for each class and pick the most likely one.
- **Naive assumption:** Words are conditionally independent given the class. This is clearly false (e.g., "not" and "good" are dependent) but works surprisingly well in practice.
- **Multinomial distribution:** Each document is treated as a bag-of-words count vector; P(document|class) = product over all word positions of P(word|class).
- **Laplace smoothing:** Add α to all word counts to avoid zero probabilities for unseen words. P(word|class) = (count(word, class) + α) / (total_words(class) + α × vocab_size).
- **Log-space computation:** Multiplying many probabilities underflows; instead we sum log-probabilities: log P(class|doc) = log P(class) + Σ log P(word|class).

In [ ]:
# Load IMDB dataset from public GitHub mirror (no kagglehub needed)
url = "https://raw.githubusercontent.com/laxmimerit/All-CSV-ML-Data-Files-Download/master/IMDB-Dataset.csv"
print(f"Loading IMDB dataset from GitHub mirror...")
df = pd.read_csv(url, encoding='utf-8')

# Quick data inspection
print(f"\nDataset shape: {df.shape}\n")
print("--- First 5 rows ---")
print(df.head())
print("\n--- Dataset info ---")
df.info()
print("\n--- Target distribution ---")
print(df['sentiment'].value_counts())

# The dataset has 50,000 movie reviews with two columns:
#   review:     Full text of the movie review
#   sentiment:  Binary label — 'positive' or 'negative'

In [ ]:
# ------ Handle nulls ------
print(f"Null values before cleaning: {df.isnull().sum().sum()}")

# ------ Encode labels ------
label_map = {'positive': 1, 'negative': 0}
y = np.array([label_map[s] for s in df['sentiment']])

# ------ Text cleaning function ------
def clean_text(text):
    """Lowercase, remove HTML tags, remove punctuation, collapse whitespace."""
    text = text.lower()
    text = re.sub(r'<br\\s*/?>', ' ', text)   # remove <br> tags
    text = re.sub(r'[^a-z\\s]', '', text)       # keep only letters and spaces
    text = re.sub(r'\\s+', ' ', text).strip()    # collapse whitespace
    return text

print("Cleaning text...")
df['clean_review'] = df['review'].apply(clean_text)

# ------ Train / Test Split (stratified 80/20) ------
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['review'], y, test_size=0.2, random_state=42, stratify=y
)
X_train_clean, X_test_clean, _, _ = train_test_split(
    df['clean_review'], y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {len(X_train_text)} | Test size: {len(X_test_text)}")
print(f"Train positive ratio: {y_train.mean():.3f} | Test positive ratio: {y_test.mean():.3f}")

## Part 2: From-Scratch Implementation — Multinomial Naive Bayes

We implement **Multinomial Naive Bayes from scratch using only NumPy**. The model learns class priors and word probabilities from the training data, then classifies new documents by computing log-posteriors and picking the most probable class.

**How it works:**
1. Build a bag-of-words count matrix where each row is a document and each column is a word in the vocabulary.
2. Compute P(class) = count of documents in class / total documents (log-space).
3. Compute P(word|class) = (count of word in class + α) / (total words in class + α × vocab_size) (log-space).
4. For a new document, sum log P(word|class) over all words present, add log P(class), and pick the class with the highest log-posterior.

We use log-space throughout to avoid floating-point underflow when multiplying many small probabilities.

In [ ]:
class MultinomialNaiveBayesScratch:
    """
    From-scratch Multinomial Naive Bayes using only NumPy.

    Learns P(class) and P(word|class) from bag-of-words counts.
    Uses log-space computation for numerical stability.
    Laplace smoothing (alpha) prevents zero probabilities.
    """
    def __init__(self, alpha=1.0):
        # INTERVIEW NOTE: alpha=1 is Laplace smoothing; alpha<1 is Lidstone smoothing.
        # Higher alpha = stronger smoothing = more weight to uniform prior.
        self.alpha = alpha

    def fit(self, X, y):
        """
        X: numpy array of shape (n_samples, n_features) — bag-of-words counts
        y: numpy array of shape (n_samples,) — class labels (0 or 1)
        """
        n_samples, n_features = X.shape
        self.classes_ = np.unique(y)
        n_classes = len(self.classes_)

        # Class prior: P(class) = count(class) / total
        # INTERVIEW NOTE: Log-priors avoid underflow in probability products
        class_counts = np.array([(y == c).sum() for c in self.classes_])
        self.class_log_prior_ = np.log(class_counts / n_samples)

        # Feature log probabilities: log P(word|class)
        self.feature_log_prob_ = np.zeros((n_classes, n_features))

        for i, c in enumerate(self.classes_):
            X_c = X[y == c]                                   # documents in class c
            total_words = X_c.sum()                            # total word count in class c
            feature_counts = X_c.sum(axis=0) + self.alpha      # per-word count + smoothing
            # INTERVIEW NOTE: Laplace smoothing adds alpha to EVERY feature count,
            # ensuring no P(word|class) is ever zero.
            self.feature_log_prob_[i] = np.log(feature_counts) - np.log(total_words + self.alpha * n_features)

        return self

    def predict_log_proba(self, X):
        """Return log-probabilities for each class."""
        # log P(class|doc) = log P(class) + sum over words of log P(word|class)
        # INTERVIEW NOTE: In log-space, multiplication becomes addition — this is the
        # key numerical trick that makes Naive Bayes stable.
        return self.class_log_prior_ + X @ self.feature_log_prob_.T

    def predict_proba(self, X):
        """Return probabilities for each class (normalized)."""
        log_probs = self.predict_log_proba(X)
        # Subtract max for numerical stability before exponentiating
        log_probs = log_probs - log_probs.max(axis=1, keepdims=True)
        probs = np.exp(log_probs)
        return probs / probs.sum(axis=1, keepdims=True)

    def predict(self, X):
        """Predict class (0 or 1) for each document."""
        log_probs = self.predict_log_proba(X)
        return self.classes_[np.argmax(log_probs, axis=1)]

In [ ]:
# ------ Build bag-of-words matrix for scratch model ------
print("Building CountVectorizer (max_features=5000) for scratch model...")
vec_cv = CountVectorizer(max_features=5000)
X_train_counts = vec_cv.fit_transform(X_train_clean).toarray()
X_test_counts = vec_cv.transform(X_test_clean).toarray()
print(f"  Train BoW matrix: {X_train_counts.shape}")
print(f"  Test  BoW matrix: {X_test_counts.shape}")

# ------ Train from-scratch model ------
print("\nTraining from-scratch Multinomial Naive Bayes...")
scratch_nb = MultinomialNaiveBayesScratch(alpha=1.0)
scratch_nb.fit(X_train_counts, y_train)

# ------ Evaluate ------
y_pred_scratch = scratch_nb.predict(X_test_counts)

acc_s  = accuracy_score(y_test, y_pred_scratch)
prec_s = precision_score(y_test, y_pred_scratch)
rec_s  = recall_score(y_test, y_pred_scratch)
f1_s   = f1_score(y_test, y_pred_scratch)

print("\n" + "=" * 52)
print("  FROM-SCRATCH MULTINOMIAL NAIVE BAYES RESULTS")
print("=" * 52)
print(f"  Accuracy : {acc_s:.4f}")
print(f"  Precision: {prec_s:.4f}")
print(f"  Recall   : {rec_s:.4f}")
print(f"  F1 Score : {f1_s:.4f}")
print("=" * 52)

## Part 3: Sklearn Implementation — MultinomialNB & Logistic Regression

sklearn provides two relevant tools for text classification:

1. **CountVectorizer + MultinomialNB** — The exact same algorithm as our scratch implementation, but with optimized Cython internals, built-in cross-validation, and production-grade numerical stability.
2. **TfidfVectorizer + LogisticRegression** — A different paradigm: TF-IDF weighting down-weights common words and emphasizes discriminative terms. Logistic Regression models P(class|document) directly (discriminative) rather than P(document|class) (generative).

Comparing both against our scratch version reveals the practical benefits of sklearn's optimized routines and the different inductive biases of generative vs. discriminative classifiers.

In [ ]:
# ------ Method A: CountVectorizer + MultinomialNB ------
print("Method A: CountVectorizer + MultinomialNB...")
vec_a = CountVectorizer(max_features=5000)
X_train_a = vec_a.fit_transform(X_train_clean)
X_test_a = vec_a.transform(X_test_clean)

sk_nb = MultinomialNB(alpha=1.0)
sk_nb.fit(X_train_a, y_train)
y_pred_nb = sk_nb.predict(X_test_a)

acc_nb  = accuracy_score(y_test, y_pred_nb)
prec_nb = precision_score(y_test, y_pred_nb)
rec_nb  = recall_score(y_test, y_pred_nb)
f1_nb   = f1_score(y_test, y_pred_nb)

# ------ Method B: TfidfVectorizer + LogisticRegression ------
print("Method B: TfidfVectorizer + LogisticRegression...")
vec_b = TfidfVectorizer(max_features=5000, stop_words='english', sublinear_tf=True)
X_train_b = vec_b.fit_transform(X_train_clean)
X_test_b = vec_b.transform(X_test_clean)

lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
lr.fit(X_train_b, y_train)
y_pred_lr = lr.predict(X_test_b)

acc_lr  = accuracy_score(y_test, y_pred_lr)
prec_lr = precision_score(y_test, y_pred_lr)
rec_lr  = recall_score(y_test, y_pred_lr)
f1_lr   = f1_score(y_test, y_pred_lr)

# ------ Comparison Table ------
print("\n" + "=" * 75)
print("  MODEL COMPARISON")
print("=" * 75)
print(f"  {'Metric':<15} {'Scratch NB':<18} {'Sklearn MNB':<18} {'Tfidf+LR':<18}")
print(f"  {'-'*15} {'-'*18} {'-'*18} {'-'*18}")
print(f"  {'Accuracy':<15} {acc_s:<18.4f} {acc_nb:<18.4f} {acc_lr:<18.4f}")
print(f"  {'Precision':<15} {prec_s:<18.4f} {prec_nb:<18.4f} {prec_lr:<18.4f}")
print(f"  {'Recall':<15} {rec_s:<18.4f} {rec_nb:<18.4f} {rec_lr:<18.4f}")
print(f"  {'F1 Score':<15} {f1_s:<18.4f} {f1_nb:<18.4f} {f1_lr:<18.4f}")
print("=" * 75)

# Classification reports
print("\nScratch NB -- Classification Report:")
print(classification_report(y_test, y_pred_scratch, target_names=['negative', 'positive']))
print("\nSklearn MultinomialNB -- Classification Report:")
print(classification_report(y_test, y_pred_nb, target_names=['negative', 'positive']))
print("\nTfidf + LogisticRegression -- Classification Report:")
print(classification_report(y_test, y_pred_lr, target_names=['negative', 'positive']))

In [ ]:
# ---- Visualization 1: Confusion Matrices ----
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cm_scratch = confusion_matrix(y_test, y_pred_scratch)
cm_sklearn = confusion_matrix(y_test, y_pred_nb)

sns.heatmap(cm_scratch, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=axes[0])
axes[0].set_title('Scratch NB -- Confusion Matrix', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_sklearn, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Negative','Positive'],
            yticklabels=['Negative','Positive'], ax=axes[1])
axes[1].set_title('Sklearn MultinomialNB -- Confusion Matrix', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# ---- Visualization 2: Bar Chart Comparison ----
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
scratch_scores = [acc_s, prec_s, rec_s, f1_s]
sklearn_scores = [acc_nb, prec_nb, rec_nb, f1_nb]
lr_scores     = [acc_lr, prec_lr, rec_lr, f1_lr]

x = np.arange(len(metrics_names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width, scratch_scores, width, label='Scratch NB', color='#2c3e50')
bars2 = ax.bar(x, sklearn_scores, width, label='Sklearn MultinomialNB', color='#27ae60')
bars3 = ax.bar(x + width, lr_scores, width, label='Tfidf + LogisticRegression', color='#8e44ad')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_names)
ax.legend(loc='lower right')
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars3:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

## Part 4: Hyperparameter Experiments

Two critical hyperparameters for Multinomial Naive Bayes:

1. **Smoothing alpha (α)** — Controls how much probability mass is reserved for unseen words. α = 0 gives no smoothing (zero probabilities for unseen words), α = 1 is Laplace smoothing, α < 1 is Lidstone smoothing. Higher α = stronger uniform prior on the vocabulary.
2. **max_features** — Controls vocabulary size. A small vocabulary may miss discriminative words; a large vocabulary increases sparsity and computational cost.

We vary both using 3-fold cross-validation with our scratch implementation and the CountVectorizer output.

In [ ]:
print("Hyperparameter sweep -- this may take a minute...\n")

# Build CV base matrix (fixed max_features=5000 for alpha experiment)
vec_cv_base = CountVectorizer(max_features=5000)
X_cv_base = vec_cv_base.fit_transform(X_train_clean).toarray()

# ---- Experiment 1: Vary alpha ----
alphas = [0.01, 0.1, 1.0, 10.0]
print("Varying smoothing alpha (max_features=5000, 3-fold CV)...")

alpha_means, alpha_stds = [], []
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for alpha in alphas:
    fold_accs = []
    for train_i, val_i in skf.split(X_cv_base, y_train):
        X_tr, X_va = X_cv_base[train_i], X_cv_base[val_i]
        y_tr, y_va = y_train[train_i], y_train[val_i]
        m = MultinomialNaiveBayesScratch(alpha=alpha)
        m.fit(X_tr, y_tr)
        preds = m.predict(X_va)
        fold_accs.append(accuracy_score(y_va, preds))
    alpha_means.append(np.mean(fold_accs))
    alpha_stds.append(np.std(fold_accs))
    print(f"  alpha={alpha:5.2f}  ->  CV Accuracy: {alpha_means[-1]:.4f} +/- {alpha_stds[-1]:.4f}")

# ---- Experiment 2: Vary max_features (fixed alpha=1.0) ----
max_features_list = [1000, 5000, 10000]
print("\nVarying max_features (alpha=1.0, 3-fold CV)...")

feat_means, feat_stds = [], []

for mf in max_features_list:
    vec_cv_mf = CountVectorizer(max_features=mf)
    X_cv_mf = vec_cv_mf.fit_transform(X_train_clean).toarray()
    fold_accs = []
    for train_i, val_i in skf.split(X_cv_mf, y_train):
        X_tr, X_va = X_cv_mf[train_i], X_cv_mf[val_i]
        y_tr, y_va = y_train[train_i], y_train[val_i]
        m = MultinomialNaiveBayesScratch(alpha=1.0)
        m.fit(X_tr, y_tr)
        preds = m.predict(X_va)
        fold_accs.append(accuracy_score(y_va, preds))
    feat_means.append(np.mean(fold_accs))
    feat_stds.append(np.std(fold_accs))
    print(f"  max_features={mf:5d}  ->  CV Accuracy: {feat_means[-1]:.4f} +/- {feat_stds[-1]:.4f}")

# ---- Plot results ----
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Alpha experiment
axes[0].errorbar([str(a) for a in alphas], alpha_means, yerr=alpha_stds,
                 marker='o', capsize=6, color='#2c3e50', linewidth=2, markersize=8)
axes[0].set_xlabel('Smoothing Alpha (α)', fontsize=11)
axes[0].set_ylabel('CV Accuracy', fontsize=11)
axes[0].set_title('Effect of Alpha on Accuracy', fontweight='bold')
axes[0].grid(alpha=0.3)
axes[0].set_ylim(0.5, 1.0)
for i, (a, m, s) in enumerate(zip(alphas, alpha_means, alpha_stds)):
    axes[0].annotate(f'{m:.3f}', (i, m), textcoords='offset points',
                     xytext=(0, 12), ha='center', fontsize=9)

# max_features experiment
axes[1].errorbar([str(m) for m in max_features_list], feat_means, yerr=feat_stds,
                 marker='s', capsize=6, color='#8e44ad', linewidth=2, markersize=8)
axes[1].set_xlabel('Max Features (Vocabulary Size)', fontsize=11)
axes[1].set_ylabel('CV Accuracy', fontsize=11)
axes[1].set_title('Effect of Vocabulary Size on Accuracy', fontweight='bold')
axes[1].grid(alpha=0.3)
axes[1].set_ylim(0.5, 1.0)
for i, (m, mean, std) in enumerate(zip(max_features_list, feat_means, feat_stds)):
    axes[1].annotate(f'{mean:.3f}', (i, mean), textcoords='offset points',
                     xytext=(0, 12), ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## Part 5: Interview Corner

### Q: Why does Naive Bayes work so well for text classification even though its core assumption (conditional independence of words) is clearly violated?

**Short answer:** Naive Bayes does not need well-calibrated probabilities — it only needs the *ranking* of classes to be correct. The independence assumption cancels out in surprising ways, and the model's high bias prevents overfitting on text data.

**Narrative explanation:**

Let's examine the key reasons:

1. **Ranking, not calibration.** Naive Bayes is a *poor probability estimator* but a *good classifier*. Even if P(class|document) is off by orders of magnitude, the *relative* ordering between classes (argmax) often remains correct. The independence assumption distorts absolute probabilities but preserves relative rankings surprisingly well.

2. **Dependencies cancel out across classes.** When two words are correlated (e.g., "not" and "good" in "not good"), the violation affects both P(positive|document) and P(negative|document) similarly. Since we only care about which is larger, the errors cancel in the ratio — this is known as the *"cancellation effect"* in the literature.

3. **Feature competition via log-odds.** The decision rule is essentially a linear model in log-space: log P(positive|doc) − log P(negative|doc) = log(odds) + Σ word_i × (log P(word_i|positive) − log P(word_i|negative)). Strongly discriminative words (e.g., "awful", "amazing") get high log-odds weights, while neutral words contribute near zero — the model naturally focuses on signal.

4. **Bias-variance tradeoff.** The independence assumption introduces *bias* (the model is wrong), but reduces *variance* (the model is stable). On high-dimensional sparse text data, variance is the dominant source of error — the bias from Naive Bayes is a small price to pay for dramatically lower variance. More complex models (e.g., unregularized logistic regression) often *overfit* on the same data.

5. **Empirical confirmation.** Hundreds of papers over three decades have confirmed that Naive Bayes is competitive on text classification despite (or because of) its simplicity. It remains a standard baseline that more complex models must beat — and sometimes they barely do.

## Key Takeaways — Placement Interview Revision

1. **Naive Bayes applies Bayes' theorem with a strong independence assumption:** P(class|document) ∝ P(class) × Π P(word|class). Despite being "naive," it performs well on high-dimensional sparse text data.

2. **Laplace smoothing (α = 1) is essential:** Without smoothing, any unseen word in a test document would make P(document|class) = 0, breaking the classifier. Alpha controls the strength of the uniform prior over the vocabulary.

3. **Log-space computation prevents underflow:** Multiplying hundreds of small probabilities (each < 1.0) underflows to zero in floating point. Summing log-probabilities is numerically stable and mathematically equivalent.

4. **Scratch vs. sklearn matches closely:** A correct NumPy implementation of MultinomialNB should produce nearly identical results to sklearn's optimized version, confirming algorithmic equivalence.

5. **Tfidf + LogisticRegression is a stronger baseline:** TF-IDF weighting dampens frequent, uninformative words, and Logistic Regression's discriminative training often outperforms the generative Naive Bayes on sentiment classification.